In [ ]:
#!/usr/bin/env python3
"""
Summarize regression metrics for tea_split3_all_celltypes.
Retained methods: DAIF-CKA (K=1), DAIF-CKA (K=3), JAFAR, CoopReg, MOFA+, Multigrate.
Note: OrchAMP = DAIF-CKA (K=1); EB-PCA = DAIF-CKA (K=3).
Shows: RMSE, Pearson R², Spearman r.
"""

from __future__ import annotations
import json
import re
from pathlib import Path
import pandas as pd

# ---- Settings ----
dataset_name = "tea"
split_tag    = "tea_split3_all_celltypes"

results_root = Path("../results") / dataset_name
out_dir      = results_root / f"{split_tag}_reg_summary"
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / f"{split_tag}_combined_reg_metrics.csv"
tex_path = out_dir / f"{split_tag}_combined_reg_metrics.tex"

# ---- Folders to include and their display names ----
# Ordered as desired in the output table
INCLUDED_FOLDERS = [
    ("amp_early_fusion_reg_CD45RA_linear",        "DAIF-CKA (K=1)"),
    ("amp_late_fusion_reg_CD45RA_linear",          "DAIF-CKA (K=3)"),
    ("jafar_CD45RA",                               "JAFAR"),
    ("coopreg_CD45RA",                             "Co-operative Regression"),
    ("mofa_reg_CD45RA",                            "MOFA+"),
    ("multigrate_ols_CD45RA",                      "Multigrate (OLS)"),
]

def display_name(folder_suffix: str, d: dict, label_override) -> str:
    if label_override is not None:
        return label_override
    run = d.get("run", {})
    k = run.get("num_clusters_selected") or run.get("num_clusters")
    return f"DAIF-CKA (K={k})" if k is not None else "DAIF-CKA"

# ---- Collect metrics ----
records = []
for folder_suffix, label in INCLUDED_FOLDERS:
    folder = results_root / f"{split_tag}_{folder_suffix}"
    json_path = None
    for candidate in ["amp_test_metrics.json", "test_metrics.json"]:
        p = folder / candidate
        if p.exists():
            json_path = p
            break

    if json_path is None:
        print(f"[SKIP] No JSON found: {folder}")
        continue

    with open(json_path) as f:
        d = json.load(f)

    method     = display_name(folder_suffix, d, label)
    rmse       = d.get("rmse")
    pearson_r  = d.get("pearson_r")
    spearman_r = d.get("spearman_r")
    pearson_r2 = pearson_r**2 if pearson_r is not None else None

    records.append({
        "Method":     method,
        "RMSE":       rmse,
        "Pearson R²": pearson_r2,
        "Spearman r": spearman_r,
    })
    spr = f"{spearman_r:.4f}" if spearman_r is not None else "N/A"
    pr2 = f"{pearson_r2:.4f}" if pearson_r2 is not None else "N/A"
    print(f"[OK] {method:35s}  RMSE={rmse:.4f}  PearsonR²={pr2}  SpearmanR={spr}")

df = pd.DataFrame.from_records(records).set_index("Method")

df.to_csv(csv_path)
print(f"\n[Saved] CSV → {csv_path}")

# ---- LaTeX ----
cols = ["RMSE", "Pearson R²", "Spearman r"]
lower_better = {"RMSE"}
higher_better = {"Pearson R²", "Spearman r"}

def best_mask(df, cols):
    mask = {c: set() for c in cols}
    for c in cols:
        col = df[c].dropna()
        if col.empty:
            continue
        best = col.min() if c in lower_better else col.max()
        mask[c] = set(df.index[df[c] == best])
    return mask

mask = best_mask(df, cols)

lines = [
    "% Requires: \\usepackage{booktabs}",
    "\\begin{table}[!htbp]",
    "\\centering",
    "\\small",
    "\\begin{tabular}{lrrr}",
    "\\toprule",
    "Method & RMSE & Pearson $R^2$ & Spearman $r$ \\\\",
    "\\midrule",
]
for idx in df.index:
    row = [idx]
    for c in cols:
        v = df.at[idx, c]
        if pd.isna(v):
            cell = "--"
        else:
            cell = f"{float(v):.4f}"
            if idx in mask.get(c, set()):
                cell = f"\\textbf{{{cell}}}"
        row.append(cell)
    lines.append(" & ".join(row) + " \\\\")
lines += [
    "\\bottomrule",
    "\\end{tabular}",
    "\\caption{CD45RA prediction on TEA-seq (Split 3, all cell types). "
    "Lower RMSE is better; higher Pearson $R^2$ and Spearman $r$ are better.}",
    "\\label{tab:tea_regression_split3}",
    "\\end{table}",
]
latex = "\n".join(lines)

with open(tex_path, "w") as f:
    f.write(latex)
print(f"[Saved] LaTeX → {tex_path}")

print("\n--- Summary Table ---")
print(df.to_string(float_format="{:.4f}".format))


In [ ]:
# ------------------------------------------------------------
# Per-cell-type R² breakdown (OrcHAMP methods for this split)
# ------------------------------------------------------------
ct_csv_files = sorted(results_root.glob(f"{split_tag}_amp_*_reg_*/amp_test_metrics_by_celltype.csv"))

all_ct = []
for cpath in ct_csv_files:
    folder = cpath.parent.name
    method = folder
    if method.startswith(split_tag + "_"):
        method = method[len(split_tag) + 1:]
    for suffix in ["_CD45RA_linear", "_CD45RA", "_linear"]:
        if method.endswith(suffix):
            method = method[: -len(suffix)]
            break
    df_ct = pd.read_csv(cpath, index_col=0)
    df_ct["method"] = method
    all_ct.append(df_ct)
    print(f"[CT] loaded: {method} ({len(df_ct)} rows)")

if all_ct:
    df_all_ct = pd.concat(all_ct)
    r2_pivot = df_all_ct.pivot_table(index=df_all_ct.index, columns="method", values="r2")
    r2_pivot.index.name = "cell_type"
    print("\n--- R² per cell type (OrcHAMP methods) ---")
    print(r2_pivot.to_string())

    ct_path = out_dir / f"{split_tag}_amp_celltype_r2_summary.csv"
    r2_pivot.to_csv(ct_path)
    print(f"\n[Saved] Cell-type R² summary → {ct_path}")

    full_path = out_dir / f"{split_tag}_amp_celltype_full_metrics.csv"
    df_all_ct.to_csv(full_path)
    print(f"[Saved] Full cell-type metrics  → {full_path}")
else:
    print(f"[WARN] No per-cell-type CSV files found matching: {split_tag}_amp_*_reg_*/")

In [ ]:
# ============================================================
# Multi-split LaTeX table (booktabs, \scriptsize)
# Three splits × 6 methods × {RMSE, Pearson R², Spearman r}
# Note: OrchAMP = DAIF-CKA (K=1); EB-PCA = DAIF-CKA (K=3)
# ============================================================

from __future__ import annotations
import json
from pathlib import Path
import numpy as np

results_root = Path("../results/tea")
out_tex = results_root / "all_splits_reg_metrics.tex"

# ---- Split definitions ----
SPLITS = [
    ("tea_split1_tcells",        "Split 1 (T cells)"),
    ("tea_split3_all_celltypes", "Split 3 (All cell types)"),
    ("tea_split6_mixed_b",       "Split 6 (B lineage)"),
]

# ---- Methods: (folder_suffix, display_name) ----
METHODS = [
    ("amp_early_fusion_reg_CD45RA_linear",        "DAIF-CKA (K=1)"),
    ("amp_late_fusion_reg_CD45RA_linear",          "DAIF-CKA (K=3)"),
    ("jafar_CD45RA",                               "JAFAR"),
    ("coopreg_CD45RA",                             "Co-op Regression"),
    ("mofa_reg_CD45RA",                            "MOFA+"),
    ("multigrate_ols_CD45RA",                      "Multigrate (OLS)"),
]

METRICS = ["rmse", "pearson_r2", "spearman_r"]
COL_HEADERS = ["RMSE", "Pearson $R^2$", "Spearman $r$"]
LOWER_BETTER = {"rmse"}

def load_metrics(folder: Path) -> dict | None:
    for candidate in ["amp_test_metrics.json", "test_metrics.json"]:
        p = folder / candidate
        if p.exists():
            with open(p) as f:
                return json.load(f)
    return None

def get_method_name(suffix: str, label_override, d: dict) -> str:
    if label_override is not None:
        return label_override
    run = d.get("run", {})
    k = run.get("num_clusters_selected") or run.get("num_clusters")
    return f"DAIF-CKA (K={k})" if k is not None else "DAIF-CKA"

def fmt(v, bold=False) -> str:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "--"
    s = f"{v:.4f}"
    return f"\\textbf{{{s}}}" if bold else s

# ---- Build one LaTeX table per split ----
table_blocks = []

for split_tag, split_label in SPLITS:
    # collect rows: list of (method_name, rmse, pearson_r2, spearman_r)
    rows = []
    for suffix, label_override in METHODS:
        folder = results_root / f"{split_tag}_{suffix}"
        d = load_metrics(folder)
        if d is None:
            rows.append((label_override or "DAIF-CKA", None, None, None))
            continue
        name = get_method_name(suffix, label_override, d)
        rmse      = d.get("rmse")
        pr        = d.get("pearson_r")
        pr2       = pr**2 if pr is not None else None
        spearman  = d.get("spearman_r")
        rows.append((name, rmse, pr2, spearman))

    # best per column (ignore None)
    def best_val(col_idx, lower_better):
        vals = [r[col_idx] for r in rows if r[col_idx] is not None]
        if not vals:
            return None
        return min(vals) if lower_better else max(vals)

    best_rmse  = best_val(1, True)
    best_pr2   = best_val(2, False)
    best_spear = best_val(3, False)

    lines = [
        f"% --- {split_label} ---",
        "\\begin{table}[!htbp]",
        "\\centering",
        "\\scriptsize",
        "\\begin{tabular}{lrrr}",
        "\\toprule",
        f"\\multicolumn{{4}}{{l}}{{\\textit{{{split_label}}}}} \\\\",
        "\\midrule",
        "Method & RMSE & Pearson $R^2$ & Spearman $r$ \\\\",
        "\\midrule",
    ]
    for name, rmse, pr2, spear in rows:
        r_rmse  = fmt(rmse,  bold=(rmse  is not None and best_rmse  is not None and abs(rmse  - best_rmse)  < 1e-9))
        r_pr2   = fmt(pr2,   bold=(pr2   is not None and best_pr2   is not None and abs(pr2   - best_pr2)   < 1e-9))
        r_spear = fmt(spear, bold=(spear is not None and best_spear is not None and abs(spear - best_spear) < 1e-9))
        lines.append(f"{name} & {r_rmse} & {r_pr2} & {r_spear} \\\\")
    lines += [
        "\\bottomrule",
        "\\end{tabular}",
        f"\\caption{{CD45RA prediction on TEA-seq: {split_label}. "
        "Lower RMSE is better; higher Pearson $R^2$ and Spearman $r$ are better. "
        "Bold = best per column.}",
        f"\\label{{tab:tea_cd45ra_{split_tag}}}",
        "\\end{table}",
        "",
    ]
    table_blocks.append("\n".join(lines))
    print(f"[OK] {split_label}")

# ---- Write .tex file ----
header = "% Auto-generated: CD45RA regression comparison across splits\n% Requires: \\usepackage{booktabs}\n\n"
body = "\n\n".join(table_blocks)

with open(out_tex, "w") as f:
    f.write(header + body)

print(f"\n[Saved] {out_tex}")
